# 1.1 · Counting steps: how cost grows with input

*Cost and measurement* · 16 exercises

Count how many steps code takes as its input grows, and name the growth with Big O: constant, logarithmic, linear, quadratic, exponential.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-1.1) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-1.1) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-1.1)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 1.1.1 · An O(1) algorithm  `Core`

**Task.** Write `is_leap_year(year)`. Explain why its step count does not depend on the input's size.

**Example.** `is_leap_year(2024)` -> `True`; `1900` -> `False`; `2000` -> `True`

**Big O.** O(1). At most three checks, whatever the year.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-1)

In [ ]:
# 1.1.1 An O(1) algorithm
def is_leap_year(year):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_1():
    assert is_leap_year(2024)
    assert not is_leap_year(1900)
    assert is_leap_year(2000)
    assert not is_leap_year(2023)


test_ex_1_1_1()
print('1.1.1 passed')

### 1.1.2 · An O(N) algorithm  `Core`

**Task.** Write `array_sum(arr)` with a loop (no `sum()`), and say how many steps it takes for N elements.

**Example.** `[1, 2, 3, 4]` -> `10`, 4 loop steps

**Big O.** O(N). One step per element.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-2)

In [ ]:
# 1.1.2 An O(N) algorithm
def array_sum(arr):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_2():
    assert array_sum([1, 2, 3, 4]) == 10
    assert array_sum([]) == 0


test_ex_1_1_2()
print('1.1.2 passed')

### 1.1.3 · An O(log N) algorithm  `Core`

**Task.** Write `halving_steps(n)` returning how many times you can halve `n` (integer division) before it reaches 1.

**Example.** `halving_steps(8)` -> `3`; `halving_steps(1024)` -> `10`; `halving_steps(1)` -> `0`

**Big O.** O(log N). This is the definition of log2 N for powers of two.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-3)

In [ ]:
# 1.1.3 An O(log N) algorithm
def halving_steps(n):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_3():
    assert halving_steps(8) == 3
    assert halving_steps(1024) == 10
    assert halving_steps(1) == 0


test_ex_1_1_3()
print('1.1.3 passed')

### 1.1.4 · Doubling: rounds until everyone knows  `Practice`

**Task.** One person knows a rumour. Each round, everyone who knows tells one person who does not. Write `rounds_to_reach(people)` returning how many rounds pass before at least `people` know.

**Example.** `rounds_to_reach(1)` -> `0`; `rounds_to_reach(2)` -> `1`; `rounds_to_reach(1000)` -> `10`

**Big O.** O(log N). The number who know doubles each round, the mirror image of halving.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-4)

In [ ]:
# 1.1.4 Doubling: rounds until everyone knows
def rounds_to_reach(people):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_4():
    assert rounds_to_reach(1) == 0
    assert rounds_to_reach(2) == 1
    assert rounds_to_reach(3) == 2
    assert rounds_to_reach(1000) == 10
    assert rounds_to_reach(1024) == 10
    assert rounds_to_reach(1025) == 11


test_ex_1_1_4()
print('1.1.4 passed')

### 1.1.5 · Count nested-loop steps  `Practice`

**Task.** Write `all_pairs_steps(n)` (inner loop runs N times per outer step) and `unique_pairs_steps(n)` (inner loop starts at `i + 1`). Both return the number of inner-loop iterations. Classify both.

**Example.** `all_pairs_steps(4)` -> `16`; `unique_pairs_steps(4)` -> `6`

**Big O.** Both O(N^2). Unique pairs is N(N-1)/2, about half, and Big O ignores the 1/2.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-5)

In [ ]:
# 1.1.5 Count nested-loop steps
def all_pairs_steps(n):
    raise NotImplementedError


def unique_pairs_steps(n):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_5():
    assert all_pairs_steps(4) == 16
    assert unique_pairs_steps(4) == 6


test_ex_1_1_5()
print('1.1.5 passed')

### 1.1.6 · Running balance with no overdraft  `Core`

**Task.** Write `final_balance(changes)`: start at 0 and apply each deposit (positive) or withdrawal (negative) in order, skipping any withdrawal that would take the balance below 0.

**Example.** `[50, -20, -40, 10]` -> `40`

**Big O.** O(N). One pass, and each step does a constant amount of work.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-6)

In [ ]:
# 1.1.6 Running balance with no overdraft
def final_balance(changes):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_6():
    assert final_balance([50, -20, -40, 10]) == 40
    assert final_balance([]) == 0
    assert final_balance([-5]) == 0
    assert final_balance([5, -5, -1]) == 0


test_ex_1_1_6()
print('1.1.6 passed')

### 1.1.7 · Every pairing from a list  `Core`

**Task.** Write `pairings(names)` returning `'a-b'` for every pair of positions `i < j`, in that order.

**Example.** `['ana', 'bo', 'cy']` -> `['ana-bo', 'ana-cy', 'bo-cy']`

**Big O.** O(N^2): N(N-1)/2 pairings, each built once.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-7)

In [ ]:
# 1.1.7 Every pairing from a list
def pairings(names):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_7():
    assert pairings(['ana', 'bo', 'cy']) == ['ana-bo', 'ana-cy', 'bo-cy']
    assert pairings(['solo']) == []
    assert len(pairings([str(i) for i in range(10)])) == 45


test_ex_1_1_7()
print('1.1.7 passed')

### 1.1.8 · Committees of three  `Practice`

**Task.** Extend the idea to `committees(names)`: every choice of three positions `i < j < k`, written `'a,b,c'`, in that order. How does the Big O change?

**Example.** `['a', 'b', 'c', 'd']` -> `['a,b,c', 'a,b,d', 'a,c,d', 'b,c,d']`

**Big O.** O(N^3): about N^3/6 committees. Each extra nested loop over the same input adds one to the exponent.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-8)

In [ ]:
# 1.1.8 Committees of three
def committees(names):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_8():
    assert committees(['a', 'b', 'c', 'd']) == ['a,b,c', 'a,b,d', 'a,c,d', 'b,c,d']
    assert committees(['a', 'b']) == []
    assert len(committees([str(i) for i in range(10)])) == 120


test_ex_1_1_8()
print('1.1.8 passed')

### 1.1.9 · The newest entries of a log  `Core`

**Task.** Write `newest(lines, k)` returning the last `k` lines in their original order (all of them if there are fewer), reading by index rather than walking the whole list.

**Example.** `['a', 'b', 'c', 'd'], 2` -> `['c', 'd']`

**Big O.** O(k), not O(N). Reading by index jumps straight to the end, so the length of the log does not matter.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-9)

In [ ]:
# 1.1.9 The newest entries of a log
def newest(lines, k):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_9():
    assert newest(['a', 'b', 'c', 'd'], 2) == ['c', 'd']
    assert newest(['a'], 5) == ['a']
    assert newest(['a', 'b'], 0) == []


test_ex_1_1_9()
print('1.1.9 passed')

### 1.1.10 · Busiest hour of the day  `Core`

**Task.** Write `busiest_hour(times)` for a non-empty list of `'HH:MM'` strings, returning the hour (an int) with the most entries; on a tie, the earliest hour.

**Example.** `['09:15', '10:02', '09:40', '11:00']` -> `9`

**Big O.** O(N). One pass counts into 24 slots, then a fixed 24-step scan; a constant amount of extra work does not change the class.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-10) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-10)

In [ ]:
# 1.1.10 Busiest hour of the day
def busiest_hour(times):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_10():
    assert busiest_hour(['09:15', '10:02', '09:40', '11:00']) == 9
    assert busiest_hour(['10:00', '09:00']) == 9
    assert busiest_hour(['23:59', '23:00', '00:10']) == 23


test_ex_1_1_10()
print('1.1.10 passed')

### 1.1.11 · A week of slots per person  `Core`

**Task.** Write `weekly_slots(names)` returning `'<name> <day>'` for every name and each of `Mon` to `Sun`, name by name.

**Example.** `['ana']` -> `['ana Mon', 'ana Tue', 'ana Wed', 'ana Thu', 'ana Fri', 'ana Sat', 'ana Sun']`

**Big O.** O(N). The inner loop always runs 7 times, so it is 7N steps, not N^2.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-11) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-11)

In [ ]:
# 1.1.11 A week of slots per person
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


def weekly_slots(names):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_11():
    assert weekly_slots(['ana']) == ['ana Mon', 'ana Tue', 'ana Wed', 'ana Thu', 'ana Fri', 'ana Sat', 'ana Sun']
    slots = weekly_slots(['ana', 'bo'])
    assert len(slots) == 14
    assert slots[7] == 'bo Mon'
    assert weekly_slots([]) == []


test_ex_1_1_11()
print('1.1.11 passed')

### 1.1.12 · Vowels across a document  `Core`

**Task.** Write `count_vowels(lines)` counting the vowels `aeiou`, in either case, across a list of strings.

**Example.** `['Hello', 'World']` -> `3`

**Big O.** O(C) where C is the total number of characters. The loops are nested, but together they visit each character once.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-12) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-12)

In [ ]:
# 1.1.12 Vowels across a document
def count_vowels(lines):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_12():
    assert count_vowels(['Hello', 'World']) == 3
    assert count_vowels(['AEIOU xyz', '']) == 5
    assert count_vowels([]) == 0


test_ex_1_1_12()
print('1.1.12 passed')

### 1.1.13 · Is the grid symmetric?  `Core`

**Task.** Write `is_symmetric(grid)` for a square list of lists: `True` when `grid[i][j] == grid[j][i]` everywhere. Compare each pair of cells once.

**Example.** `[[1, 2], [2, 1]]` -> `True`; `[[1, 2], [3, 1]]` -> `False`

**Big O.** O(N^2) for an N x N grid. Checking only above the diagonal halves the work, and Big O ignores the half.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-13) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-13)

In [ ]:
# 1.1.13 Is the grid symmetric?
def is_symmetric(grid):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_13():
    assert is_symmetric([[1, 2], [2, 1]])
    assert not is_symmetric([[1, 2], [3, 1]])
    assert is_symmetric([])
    assert is_symmetric([[7]])
    assert is_symmetric([[1, 2, 3], [2, 5, 6], [3, 6, 9]])
    assert not is_symmetric([[1, 2, 3], [2, 5, 6], [3, 0, 9]])


test_ex_1_1_13()
print('1.1.13 passed')

### 1.1.14 · Closest pair of readings  `Core`

**Task.** Write `closest_gap(nums)` for a list of at least two numbers, returning the smallest absolute difference between any two of them, by comparing every pair.

**Example.** `[10, 4, 15, 7]` -> `3`

**Big O.** O(N^2): every pair is compared once.

**Note.** Sorting first brings this to O(N log N) (3.2.4).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-14) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-14)

In [ ]:
# 1.1.14 Closest pair of readings
def closest_gap(nums):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_14():
    assert closest_gap([10, 4, 15, 7]) == 3
    assert closest_gap([1, 100]) == 99
    assert closest_gap([5, 5, 9]) == 0
    assert closest_gap([-3, 8, -1]) == 2


test_ex_1_1_14()
print('1.1.14 passed')

### 1.1.15 · Free hours two people share  `Practice`

**Task.** Write `shared_hours(a, b)` returning the hours in `a` that also appear in `b`, in `a`'s order, using nested loops.

**Example.** `[9, 10, 13, 15]`, `[10, 11, 15]` -> `[10, 15]`

**Big O.** O(N * M) with nested loops. With two different inputs, name both sizes instead of writing O(N^2).

**Note.** Putting one list in a hash set makes it O(N + M) (Chapter 4.1).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-15) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-15)

In [ ]:
# 1.1.15 Free hours two people share
def shared_hours(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_15():
    assert shared_hours([9, 10, 13, 15], [10, 11, 15]) == [10, 15]
    assert shared_hours([], [1]) == []
    assert shared_hours([8], []) == []
    assert shared_hours([7, 8], [8, 7]) == [7, 8]


test_ex_1_1_15()
print('1.1.15 passed')

### 1.1.16 · Every on/off switch setting  `Practice`

**Task.** Write `switch_settings(n)` returning every setting of `n` on/off switches as a string of `'0'` and `'1'`, in increasing binary order. Build the strings yourself rather than with a formatting helper.

**Example.** `switch_settings(2)` -> `['00', '01', '10', '11']`

**Big O.** O(2^N): each extra switch doubles the number of settings (and each setting takes N characters to write).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-1-1-16) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-1-1-16)

In [ ]:
# 1.1.16 Every on/off switch setting
def switch_settings(n):
    raise NotImplementedError

In [ ]:
def test_ex_1_1_16():
    assert switch_settings(2) == ['00', '01', '10', '11']
    assert switch_settings(0) == ['']
    assert switch_settings(1) == ['0', '1']
    three = switch_settings(3)
    assert len(three) == 8
    assert three[0] == '000'
    assert three[5] == '101'
    assert three[-1] == '111'


test_ex_1_1_16()
print('1.1.16 passed')